# 🎮 GTA 6 Viral Clipper — Google Colab Edition
Gere cortes verticais virais automáticos para Reels, Shorts e TikTok direto na nuvem do Google!

### ⚡ Benefícios no Google Colab:
- **GPU Grátis (NVIDIA T4):** Transcrição com Whisper até 10x mais rápida do que no computador local.
- **Internet Gigabit:** Downloads parciais ultrarrápidos direto dos servidores do YouTube/Twitch.
- **Zero consumo local:** Não esquenta seu PC, não usa memória nem espaço em disco local.
- **Google Drive Integrado:** Todos os vídeos são salvos diretamente na sua conta do Google Drive.

In [ ]:
#@title 📁 1. Conectar ao Google Drive
#@markdown Clique no Play. Se abrir a janela de permissão, selecione sua conta Google e clique em Permitir.
import os
try:
    from google.colab import auth
    auth.authenticate_user()
except Exception as e:
    print('Aviso auth:', e)

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    print('✓ Google Drive montado com sucesso!')
except Exception as e:
    print('Erro ao montar:', e)
    print('DICA: Se der erro de credencial, clique no ícone de Pasta (Arquivos) na barra lateral esquerda e depois no botão com o logo do Google Drive para montar graficamente!')

In [ ]:
#@title ⚙️ 2. Instalar Dependências & Atualizar Projeto
#@markdown Instala FFmpeg, fontes Montserrat, atualiza yt-dlp e sincroniza o código oficial.
!apt-get update -qq && apt-get install -y -qq ffmpeg fonts-montserrat
!pip install -q -U --pre "yt-dlp[default]"
!pip install -q faster-whisper opencv-python-headless numpy certifi static-ffmpeg deno

!rm -rf /content/meu-gta6-clipper
!git clone https://github.com/papaigato/meu-gta6-clipper.git /content/meu-gta6-clipper
%cd /content/meu-gta6-clipper
print('✓ Ambiente configurado e 100% atualizado!')

In [ ]:
#@title 🍪 2.1 (Opcional) Subir cookies do YouTube
#@markdown Se o YouTube bloquear o IP do Colab com pedido de login, suba seu arquivo cookies.txt aqui uma vez só (ele fica salvo no seu Google Drive para sempre):
import shutil, os
from pathlib import Path

drive_cookie = Path('/content/drive/MyDrive/cookies.txt')
if drive_cookie.exists():
    print('✓ Cookies já encontrados no seu Google Drive! Não precisa subir de novo.')
else:
    try:
        from google.colab import files
        print('Selecione o arquivo cookies.txt do seu computador:')
        uploaded = files.upload()
        for fn in uploaded.keys():
            if fn.endswith('.txt'):
                shutil.copyfile(fn, '/content/drive/MyDrive/cookies.txt')
                shutil.copyfile(fn, '/content/cookies.txt')
                print('✓ Cookies salvos no Google Drive com sucesso!')
    except Exception as e:
        print('Nenhum cookie enviado (tentando modo automático com cliente Android/Web):', e)

In [ ]:
#@title 🎬 3. Gerar Cortes da Live
#@markdown Escolha o link da live, a quantidade e o estilo visual desejado:

LINK_DA_LIVE = "https://www.youtube.com/watch?v=X2rtbJJEvVE" #@param {type:"string"}
QUANTIDADE_DE_CORTES = 30 #@param {type:"integer"}
ESTILO_VISUAL = "Split-Screen (Webcam no topo + Jogo)" #@param ["Split-Screen (Webcam no topo + Jogo)", "Fundo Desfocado (Blur - Estilo Video 8)"]
PASTA_DESTINO_DRIVE = "/content/drive/MyDrive/GTA6_Cortes" #@param {type:"string"}

import os
os.environ["CLIPPER_CORTES_DIR"] = PASTA_DESTINO_DRIVE
os.environ["CLIPPER_ESTILO"] = "blur" if "Blur" in ESTILO_VISUAL else "split"

print(f"🚀 Link da Live: {LINK_DA_LIVE}")
print(f"🎨 Estilo: {os.environ['CLIPPER_ESTILO']}")
print(f"📁 Destino: {PASTA_DESTINO_DRIVE}")

!python scripts/clip.py automatico "$LINK_DA_LIVE" --quantos $QUANTIDADE_DE_CORTES

In [ ]:
#@title 📋 4. Verificar Cortes Salvos no Google Drive
import glob, os
videos = glob.glob(f"{PASTA_DESTINO_DRIVE}/*/*.mp4")
print(f"🎬 Total de vídeos prontos no Google Drive: {len(videos)}")
for v in sorted(videos)[-15:]:
    mb = os.path.getsize(v) / (1024 * 1024)
    print(f"✓ {os.path.basename(v)} ({mb:.1f} MB)")